In [1]:

'''import kagglehub

# Download latest version
path = kagglehub.dataset_download("pythonafroz/solar-power")

print("Path to dataset files:", path)'''

'import kagglehub\n\n# Download latest version\npath = kagglehub.dataset_download("pythonafroz/solar-power")\n\nprint("Path to dataset files:", path)'

In [2]:
'''!kaggle datasets list -s "solar-power"'''

'!kaggle datasets list -s "solar-power"'

In [3]:
'''!kaggle datasets download -d pythonafroz/solar-power --unzip -p > solar_power.zip solar_power_data'''

'!kaggle datasets download -d pythonafroz/solar-power --unzip -p > solar_power.zip solar_power_data'

In [ ]:
# import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.precision', 3)

In [ ]:
# Load and explore the dataset
power_data = pd.read_csv('solar_power_data/Plant_1_Generation_Data.csv')
weather_data = pd.read_csv('solar_power_data/Plant_1_Weather_Sensor_Data.csv')

print('='*100)
print("power demand:\n", power_data.head())
print('='*100)
print('whether data:\n', weather_data.head())

In [ ]:
print('power_data shape:', power_data.shape)
print('weather_data shape:', weather_data.shape)

In [ ]:
print("-"*100)
print("null values in power_data:\n", power_data.isnull().sum())
print("-"*100)
print("null values in weather_data:\n", weather_data.isnull().sum())
print("-"*100)

In [ ]:
print("data types in power_data:\n", power_data.dtypes)
print("-"*100)
print("data types in weather_data:\n", weather_data.dtypes)
print("-"*100)

In [ ]:
# convert the 'DATE_TIME' column to datetime format
power_data['DATE_TIME'] = pd.to_datetime(power_data['DATE_TIME'],  errors='coerce')
weather_data['DATE_TIME'] = pd.to_datetime(weather_data['DATE_TIME'], errors='coerce')

In [ ]:
print("data types in power_data:\n", power_data.dtypes)
print("-"*100)
print("data types in weather_data:\n", weather_data.dtypes)
print("-"*100)

In [ ]:
power_data['DATE_TIME'].head()

In [ ]:
weather_data['DATE_TIME'].head()

In [ ]:
# check unique values in each column
for col in power_data.columns:
    unique_values = power_data[col].nunique()
    print(f"Column '{col}' has {unique_values} unique values.")

print("-"*100)
for col in weather_data.columns:
    unique_values = weather_data[col].nunique()
    print(f"Column '{col}' has {unique_values} unique values.")

In [ ]:
df_solar = pd.merge(power_data.drop(columns = ['PLANT_ID']), weather_data.drop(columns = ['PLANT_ID', 'SOURCE_KEY']), on='DATE_TIME')

In [ ]:
df_solar.sample(5).style.background_gradient('coolwarm')

In [ ]:
df_solar.shape

In [ ]:
df_solar.isnull().sum()

In [ ]:
#extract date features from date_time column
df_solar['date'] = df_solar['DATE_TIME'].dt.date
df_solar['month'] = df_solar['DATE_TIME'].dt.month
df_solar['time'] = df_solar['DATE_TIME'].dt.time
df_solar['day'] = df_solar['DATE_TIME'].dt.day
df_solar['week'] = df_solar['DATE_TIME'].dt.isocalendar().week

In [ ]:
df_solar.sample(5)

In [ ]:
# add hour and minute features
df_solar['hour'] = df_solar['DATE_TIME'].dt.hour
df_solar['minute'] = df_solar['DATE_TIME'].dt.minute
df_solar['total_minutes_pass'] = df_solar['hour'] * 60 + df_solar['minute']

# add date as string column
df_solar['date_str'] = df_solar['date'].astype(str)
df_solar['hours_str'] = df_solar['hour'].astype(str)
df_solar['time_str'] = df_solar['time'].astype(str)

In [ ]:
df_solar.sample(5)

***Explore Data***

In [ ]:
df_solar.info()

In [ ]:
df_solar.describe().style.background_gradient('coolwarm')

- If AC_POWER mean is much lower than 75th percentile, most production is low with occasional spikes
- If IRRADIANCE std is high, solar conditions vary significantly throughout the dataset
- If TEMPERATURE max is unusually high/low

In [ ]:
df_solar.isnull().sum()

- dataset doesn't have any null values

In [ ]:
#Encode categorical variables
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df_solar['source_key_num'] = le.fit_transform(df_solar['SOURCE_KEY'])
df_solar.sample(5)

In [ ]:
# Visualize the distribution of the DC_POWER, AC_POWER, and TOTAL_YIELD, AMBIENT_TEMPERATURE columns and module_temperature column
plt.figure(figsize=(15,10))
plt.subplot(2,3,1)
sns.histplot(df_solar['DC_POWER'], kde=True, color='blue', bins=30)
plt.title('Distribution of DC_POWER')
plt.subplot(2,3,2)
sns.histplot(df_solar['AC_POWER'], kde=True, color='green', bins=30)
plt.title('Distribution of AC_POWER')
plt.subplot(2,3,3)
sns.histplot(df_solar['TOTAL_YIELD'], kde=True, color='orange', bins=30)
plt.title('Distribution of TOTAL_YIELD')
plt.subplot(2,3,4)
sns.histplot(df_solar['MODULE_TEMPERATURE'], kde=True, color='red', bins=30)
plt.title('Distribution of MODULE_TEMPERATURE')
plt.subplot(2,3,5)
sns.histplot(df_solar['AMBIENT_TEMPERATURE'], kde=True, color='purple', bins=30)
plt.title('Distribution of AMBIENT_TEMPERATURE')
plt.subplot(2,3,6)
sns.histplot(df_solar['IRRADIATION'], kde=True, color='brown', bins=30)
plt.title('Distribution of IRRADIATION')
plt.tight_layout()
plt.show()

***DC_POWER & AC_POWER:***
- Both show highly right-skewed distributions with massive peaks at zero
- Majority of readings are 0 (nighttime/low production hours)
- Sharp decline after peak, indicating concentrated production during peak sun hours
- Few outliers at high values suggest occasional peak production days
- Insight: Solar generation is highly dependent on time of day; most data points are non-productive hours

***TOTAL_YIELD:***
- Bimodal distribution with two distinct peaks
- Two bumps suggest different solar cycles or seasonal patterns
- Consistent upward trend indicates cumulative energy generation throughout the year
- Insight: Energy accumulation follows predictable daily and seasonal cycles

***MODULE_TEMPERATURE:***
- Right-skewed distribution centered around 20-35°C
- Long tail extending to 60°C indicates hot operating conditions
- Most readings cluster in moderate range
- Insight: Module temperatures rarely exceed 60°C; optimal operating range maintained most of the time

***AMBIENT_TEMPERATURE:***
- Normal/bell-curve distribution centered around 24-26°C
- Relatively symmetric with slight right skew
- Range: 20-34°C (reasonable for typical climate)
- Insight: Stable ambient conditions; temperature variation is moderate and predictable

***IRRADIATION:***
- Extreme right-skew with massive peak at 0
- Sharp drop-off indicates binary-like behavior (either sunny or not)
- Very few readings at high irradiation values
- Insight: Solar irradiance is sparse; most hours have zero or minimal radiation (nighttime dominates dataset)


In [ ]:
df_solar['date'].nunique()

***DC_power generation on per day basis***

In [ ]:
dc = df_solar.pivot_table(index='time', values='DC_POWER', columns='date')
dc

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Prepare data
at = df_solar.pivot_table(index='time', values='DC_POWER', columns='date')

# Create subplots
cols = at.columns
n_cols = 3
n_rows = (len(cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 40))
axes = axes.flatten()

for i, col in enumerate(cols):
    axes[i].plot(range(len(at[col])), at[col].values, color='Red', linewidth=1.5)
    axes[i].set_title(f'DC_power {col}', color='blue', fontsize=10)
    axes[i].set_xlabel('Time')
    axes[i].set_ylabel('DC_POWER')
    axes[i].grid(True, alpha=0.3)

# Hide empty subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

***Key Observations***

***1. Daily Generation Cycle****
- Start Time: Generation begins around 05:30-06:00 AM (sunrise)
- Peak Time: Maximum power reached between 11:00 AM - 02:00 PM (peak solar hours)
- End Time: Generation drops to zero around 18:00-19:00 PM (sunset)
- Duration: Approximately 12-13 hours of productive generation per day

***2. Power Output Characteristics****
- Typical Peak Range: 10,000-12,500 W per day
- Ramp-up: Gradual increase from sunrise (5-6 hours to reach peak)
- Peak Plateau: Power maintains high levels for 4-5 hours midday
- Ramp-down: Steeper decline from 2:00 PM to sunset (3-4 hours)
- Bell-curve shape: Consistent symmetric/near-symmetric distribution

***3. Day-to-Day Variability****
***High Production Days (May 15, 19, 23, 25, etc.):***
- Peak values: 11,000-12,500 W
- Smooth, stable curves
- Indicates clear, sunny conditions
- Minimal fluctuations during peak hours

***Medium Production Days (May 18, 20, 26, etc.):***
- Peak values: 8,000-10,000 W
- Some ripples/fluctuations during midday
- Suggests partial cloud cover or atmospheric interference
- More volatile output

***Low Production Days (May 20, June 1, 11, 17, etc.):***
- Peak values: 3,000-7,000 W
- Irregular, jagged curves
- Indicates cloudy or overcast conditions
- Significant hour-to-hour fluctuations

***4. Weather Impact****
- Clear days = Smooth, predictable power curves
- Cloudy days = Jagged, irregular patterns with sudden dips
- Partial clouds = Moderate variability with occasional spikes

***Forecasting Implications***
- Highly predictable: Bell-curve pattern repeats daily
- Seasonal consistency: All days show similar generation window
- Weather dependent: Cloud cover causes significant variance
- Non-linear ramp: Steeper decline in afternoon (2-6 PM)


In [ ]:
daily_dc = df_solar.groupby(['date'])['DC_POWER'].sum()

ax = daily_dc.sort_values(ascending=False).plot.bar(figsize=(15,7), legend=True, color='red')
plt.title('Daily DC Power Generation', color='blue')
plt.xlabel('Date', color='green')
plt.show()

- Highest average DC_POWER Generation is on: 2020-05-15
- Lowest average DC_POWER Generation is on : 2020-06-11
- This Large variation in the DC_POWER generation is due to the fault in the system or due to weather change, which needs to study further. But from this bar plot we find the day on which there is highest DC_POWER is generated and the day with the lowest DC_POWER generated.

***Irradians on daily basis***

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Prepare data
at = df_solar.pivot_table(index='time', values='IRRADIATION', columns='date')

# Create subplots
cols = at.columns
n_cols = 3
n_rows = (len(cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 40))
axes = axes.flatten()

for i, col in enumerate(cols):
    axes[i].plot(range(len(at[col])), at[col].values, color='skyblue', linewidth=1.5)
    axes[i].set_title(f'Irradiation {col}', color='blue', fontsize=10)
    axes[i].set_xlabel('Time')
    axes[i].set_ylabel('Irradiation')
    axes[i].grid(True, alpha=0.3)

# Hide empty subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
daily_irr = df_solar.groupby(['date'])['IRRADIATION'].sum()

ax = daily_irr.sort_values(ascending=False).plot.bar(figsize=(15,7), legend=True, color='skyblue')
plt.title('Daily Irradiation', color='blue')
plt.xlabel('Date', color='green')
plt.show()

***Ambient Temperature on daily basis***

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Prepare data
at = df_solar.pivot_table(index='time', values='AMBIENT_TEMPERATURE', columns='date')

# Create subplots
cols = at.columns
n_cols = 3
n_rows = (len(cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 40))
axes = axes.flatten()

for i, col in enumerate(cols):
    axes[i].plot(range(len(at[col])), at[col].values, color='darkorange', linewidth=1.5)
    axes[i].set_title(f'Ambient Temperature {col}', color='blue', fontsize=10)
    axes[i].set_xlabel('Time')
    axes[i].set_ylabel('Temperature (°C)')
    axes[i].grid(True, alpha=0.3)

# Hide empty subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
daily_at = df_solar.groupby(['date'])['AMBIENT_TEMPERATURE'].sum()

ax = daily_at.sort_values(ascending=False).plot.bar(figsize=(15,7), legend=True, color='darkorange')
plt.title('Daily Ambient Temp', color='blue')
plt.xlabel('Date', color='green')
plt.show()